<a href="https://colab.research.google.com/github/asifdf/ai_/blob/main/steel_crack_experiment_Backbone(U-Net).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 강철 균열 세그 실험 노트북

In [ ]:
# ====== CELL 1 (교체): 설치 + 마운트 + 데이터 확보 ======
import os, glob
!pip -q install segmentation-models-pytorch albumentations scikit-image gdown
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/segmentation_experiment"

# --- Steelcrack 확보 ---
if not glob.glob('/content/steel/**/Train/images/*', recursive=True):
    zp = f"{DRIVE}/steelcrack.zip"
    if not os.path.exists(zp):
        print("Drive에 zip 없음 → 공식 링크에서 다운로드")
        !gdown --fuzzy "https://drive.google.com/file/d/1UWcv2b6sZ3jkKBrQJ6Mh6nNraEy7MIbc/view" -O /content/steelcrack.zip
        !cp /content/steelcrack.zip "$DRIVE/steelcrack.zip"   # 다음엔 Drive에서 바로
        zp = "/content/steelcrack.zip"
    !unzip -q -o "$zp" -d /content/steel

# --- OmniCrack (E2 전이학습에만 필요. 지금 E0/E1엔 없어도 됨) ---
if os.path.exists(f"{DRIVE}/omnicrack30k.zip") and not glob.glob('/content/omnicrack30k/**/images/*', recursive=True):
    !unzip -q -o "$DRIVE/omnicrack30k.zip" -d /content/omnicrack30k

# --- 확인 (여기서 숫자가 0이면 안 됨) ---
print("steel Train images:", len(glob.glob('/content/steel/**/Train/images/*', recursive=True)))
print("steel Test  images:", len(glob.glob('/content/steel/**/Test/images/*',  recursive=True)))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 14.4 MB/s eta 0:00:00
Mounted at /content/drive
steel Train images: 3300
steel Test  images: 530


In [ ]:
# ====== CELL 2: 공통 코드 (데이터/모델/지표) ======================

# ====== CELL 2: 공통 코드 (데이터/모델/지표) ======
import glob, os, random, numpy as np, cv2, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
import segmentation_models_pytorch as smp
from skimage.morphology import skeletonize

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MEAN = np.array([0.485,0.456,0.406],np.float32); STD = np.array([0.229,0.224,0.225],np.float32)

# 데이터셋 레지스트리 (★폴라리티 실측: omni=0, steel=255 / recursive 글롭)
REG = {
  "omnicrack": dict(img="/content/omnicrack30k/**/images/{s}/*",
                    msk="/content/omnicrack30k/**/annotations/{s}/*",
                    split={"train":"training","val":"validation","test":"test"}, crack=0),
  "steelcrack": dict(img="/content/steel/**/{s}/images/*",
                     msk="/content/steel/**/{s}/masks/*",
                     split={"train":"Train","val":"Validation","test":"Test"}, crack=255),
}
def stem(p): return os.path.splitext(os.path.basename(p))[0]

def index(name, split, limit=None):
    r = REG[name]; s = r["split"][split]
    imgs = sorted(glob.glob(r["img"].format(s=s), recursive=True))
    masks = {stem(m): m for m in glob.glob(r["msk"].format(s=s), recursive=True)}
    pairs = [(ip, masks[stem(ip)], r["crack"]) for ip in imgs if stem(ip) in masks]
    return pairs[:limit] if limit else pairs

def read_mask(mp, crack):
    m = cv2.imread(mp, cv2.IMREAD_UNCHANGED)
    if m.ndim == 3: m = m[:,:,0]
    return (m == crack).astype(np.uint8)

def train_tf(sz):
    return A.Compose([
        A.Resize(sz,sz),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2),
        A.Affine(translate_percent=0.06, scale=(0.9,1.15), rotate=(-20,20), p=0.6),
        A.Perspective(scale=(0.02,0.08), p=0.3),
        A.OneOf([A.MotionBlur(blur_limit=9), A.GaussianBlur(blur_limit=7), A.Defocus(radius=(2,5))], p=0.5),
        A.RandomBrightnessContrast(0.35,0.35, p=0.6), A.RandomGamma((70,140), p=0.3),
        A.GaussNoise(p=0.3),
    ])
def val_tf(sz): return A.Compose([A.Resize(sz,sz)])

class DS(Dataset):
    def __init__(self, names, split, sz, aug, limit=None):
        self.items=[]
        for n in names: self.items += index(n, split, limit)
        assert self.items, f"데이터 0장! CELL 1의 장수 print를 확인하세요. ({names},{split})"
        self.sz=sz; self.tf = train_tf(sz) if aug else val_tf(sz)
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        ip,mp,cr = self.items[i]
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB); m = read_mask(mp,cr)
        o = self.tf(image=img, mask=m); img,m = o["image"], o["mask"]
        img = ((img.astype(np.float32)/255.-MEAN)/STD).transpose(2,0,1)
        return torch.from_numpy(img).float(), torch.from_numpy(m[None].astype(np.float32))

def cl(v,s): return float((v*s).sum())/float(s.sum()) if s.sum()>0 else 0.0
def img_metrics(pr, gt):
    pr=pr.astype(np.uint8); gt=gt.astype(np.uint8)
    if gt.sum()==0:
        return {"is_neg":1, "fp": int(pr.sum()>50)}
    inter=(pr&gt).sum()
    dice=2*inter/(pr.sum()+gt.sum()+1e-7); iou=inter/(pr.sum()+gt.sum()-inter+1e-7)
    rec=inter/(gt.sum()+1e-7); prec=inter/(pr.sum()+1e-7)
    sp,sg=skeletonize(pr>0),skeletonize(gt>0)
    cldice=2*cl(pr,sg)*cl(gt,sp)/(cl(pr,sg)+cl(gt,sp)+1e-7)
    return {"is_neg":0,"dice":dice,"iou":iou,"recall":rec,"precision":prec,"cldice":cldice}

print("공통 코드 로드 완료. device:", DEVICE)

공통 코드 로드 완료. device: cuda


In [ ]:
# ====== CELL 3: 학습/평가 함수 ======================
class DiceBCE(nn.Module):
    def __init__(s,w=0.5): super().__init__(); s.b=nn.BCEWithLogitsLoss(); s.w=w
    def forward(s,logit,t):
        p=torch.sigmoid(logit)
        num=2*(p*t).sum((1,2,3))+1; den=p.sum((1,2,3))+t.sum((1,2,3))+1
        return s.w*s.b(logit,t)+(1-s.w)*(1-(num/den).mean())

@torch.no_grad()
def evaluate(model, names, split, sz, save_overlays=0, tag="eval"):
    model.eval(); items = sum([index(n,split) for n in names], [])
    pos=[]; neg_fp=0; neg_n=0; worst=[]
    for ip,mp,cr in items:
        img=cv2.cvtColor(cv2.imread(ip),cv2.COLOR_BGR2RGB); gt=read_mask(mp,cr)
        x=cv2.resize(img,(sz,sz)).astype(np.float32)/255.; x=((x-MEAN)/STD).transpose(2,0,1)
        pr=(torch.sigmoid(model(torch.from_numpy(x[None]).float().to(DEVICE)))[0,0].cpu().numpy()>0.5).astype(np.uint8)
        gtr=cv2.resize(gt,(sz,sz),interpolation=cv2.INTER_NEAREST)
        m=img_metrics(pr,gtr)
        if m["is_neg"]: neg_n+=1; neg_fp+=m["fp"]
        else:
            pos.append(m); worst.append((m["dice"],ip,gtr,pr))
    agg={k:float(np.mean([p[k] for p in pos])) for k in ["dice","iou","recall","precision","cldice"]} if pos else {}
    agg["n_pos"]=len(pos); agg["n_neg"]=neg_n
    agg["fp_rate_on_clean"]=(neg_fp/neg_n) if neg_n else None  # 결함없는 이미지 오탐율
    if save_overlays:
        os.makedirs("overlays",exist_ok=True); worst.sort(key=lambda t:t[0])
        for i,(d,ip,gt,pr) in enumerate(worst[:save_overlays]):
            b=cv2.resize(cv2.imread(ip),(sz,sz))
            b[(gt&pr)==1]=[0,255,0]; b[(gt&~pr)==1]=[0,0,255]; b[(~gt&pr)==1]=[0,255,255]
            cv2.imwrite(f"overlays/{tag}_{i:02d}_dice{d:.2f}.png",b)
    return agg

def run(cfg):
    """실험 1건: 학습 → Steelcrack Test 평가 → 결과 dict 반환."""
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    lim = 300 if cfg.get("quick") else None
    tr=DS(cfg["train"], "train", cfg["imgsz"], cfg["aug"], lim)
    tl=DataLoader(tr, cfg["batch"], shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
    model=smp.Unet(cfg["encoder"], encoder_weights="imagenet", in_channels=3, classes=1).to(DEVICE)
    if cfg.get("init") and os.path.exists(cfg["init"]):
        model.load_state_dict(torch.load(cfg["init"], map_location=DEVICE)); print("  init from", cfg["init"])
    opt=torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    scaler=torch.cuda.amp.GradScaler(); lossf=DiceBCE()
    print(f"[{cfg['name']}] train {len(tr)}장 | {cfg['encoder']} imgsz{cfg['imgsz']} aug={cfg['aug']}")
    for ep in range(1, cfg["epochs"]+1):
        model.train(); tot=0
        for x,y in tl:
            x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad()
            with torch.cuda.amp.autocast(): loss=lossf(model(x),y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot+=loss.item()
        print(f"  ep{ep}/{cfg['epochs']} loss {tot/len(tl):.4f}")
    # 평가 = Steelcrack Test (믿는 자)
    ov = 10 if cfg.get("save_overlays") else 0
    res = evaluate(model, ["steelcrack"], "test", cfg["imgsz"], save_overlays=ov, tag=cfg["name"])
    path=f"{DRIVE}/runs/best_{cfg['name']}.pt"; os.makedirs(f"{DRIVE}/runs",exist_ok=True)
    torch.save(model.state_dict(), path)
    res["name"]=cfg["name"]; res["weights"]=path
    print("  → Steelcrack Test:", {k:(round(v,3) if isinstance(v,float) else v) for k,v in res.items() if k not in("name","weights")})
    return res

# 결과 기록 (드라이브 CSV에 누적)
import csv
def log_result(res):
    p=f"{DRIVE}/experiment_results.csv"; head=not os.path.exists(p)
    with open(p,"a",newline="") as f:
        w=csv.DictWriter(f, fieldnames=["name","dice","iou","recall","precision","cldice","fp_rate_on_clean","n_pos","n_neg","weights"])
        if head: w.writeheader()
        w.writerow({k:res.get(k) for k in w.fieldnames})
    print("logged →", p)

print("학습/평가 함수 로드 완료.")

학습/평가 함수 로드 완료.


In [ ]:
import pandas as pd
_orig_run = run
def run(cfg):
    wp = f"{DRIVE}/runs/best_{cfg['name']}.pt"
    if os.path.exists(wp):
        row = {"name": cfg["name"], "weights": wp}
        csvp = f"{DRIVE}/experiment_results.csv"
        if os.path.exists(csvp):
            d = pd.read_csv(csvp); m = d[d.name == cfg["name"]]
            if len(m): row.update(m.iloc[-1].to_dict())
        print(f"[{cfg['name']}] 이미 완료 → 스킵/재사용 ({wp})")
        return row
    return _orig_run(cfg)

In [ ]:
# ====== CELL 4: E0 — 파이프라인 점검 (제일 먼저!) ======================
# quick=True 로 소량·1에폭만. 끝까지 돌아가는지 확인용. 5분 이내.
res = run(dict(name="E0_smoke", train=["steelcrack"], encoder="mobilenet_v2",
               imgsz=384, batch=8, epochs=1, lr=3e-4, aug=True, quick=True, save_overlays=True))
log_result(res)
# overlays/ 폴더의 이미지 몇 장 눈으로 확인. 여기까지 되면 파이프라인 OK.

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.2MB            

model.safetensors: downloading bytes:           |  0.00B            

[E0_smoke] train 300장 | mobilenet_v2 imgsz384 aug=True


/tmp/ipykernel_1459/438889622.py:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(); lossf=DiceBCE()
/tmp/ipykernel_1459/438889622.py:49: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): loss=lossf(model(x),y)


  ep1/1 loss 0.7156
  → Steelcrack Test: {'dice': 0.207, 'iou': 0.123, 'recall': 0.74, 'precision': 0.135, 'cldice': 0.19, 'n_pos': 530, 'n_neg': 0, 'fp_rate_on_clean': None}
logged → /content/drive/MyDrive/segmentation_experiment/experiment_results.csv


In [ ]:
# ====== CELL 5: E1 — 백본 비교 (경량 3종) ======================
# 젯슨 고려. 어떤 백본이 강철에서 성능/속도 균형 좋은지.
for enc in ["mobilenet_v2", "efficientnet-b0", "resnet34"]:
    r = run(dict(name=f"E1_{enc}", train=["steelcrack"], encoder=enc,
                 imgsz=512, batch=8, epochs=25, lr=3e-4, aug=True))
    log_result(r)

[E1_mobilenet_v2] 이미 완료 → 스킵/재사용 (/content/drive/MyDrive/segmentation_experiment/runs/best_E1_mobilenet_v2.pt)
logged → /content/drive/MyDrive/segmentation_experiment/experiment_results.csv
[E1_efficientnet-b0] 이미 완료 → 스킵/재사용 (/content/drive/MyDrive/segmentation_experiment/runs/best_E1_efficientnet-b0.pt)
logged → /content/drive/MyDrive/segmentation_experiment/experiment_results.csv
[E1_resnet34] 이미 완료 → 스킵/재사용 (/content/drive/MyDrive/segmentation_experiment/runs/best_E1_resnet34.pt)
logged → /content/drive/MyDrive/segmentation_experiment/experiment_results.csv


In [ ]:
# ====== CELL 6: E2 — 전이학습 A/B (핵심 실험) ======================
# (A) Steelcrack만  vs  (B) OmniCrack 사전학습 → Steelcrack 파인튜닝
# 도메인갭을 얼마나 메우는지 = 발표/논문 핵심 근거.
best_enc = "efficientnet-b0"   # E1 결과 보고 제일 좋은 걸로 바꿔

# (A) 강철만 (E1에서 이미 있으면 재사용 가능)
rA = run(dict(name="E2A_steelonly", train=["steelcrack"], encoder=best_enc,
              imgsz=512, batch=8, epochs=30, lr=3e-4, aug=True)); log_result(rA)

# (B-1) OmniCrack 사전학습 (오래 걸림; quick으로 줄이거나 epochs 낮춰도 됨)
rP = run(dict(name="E2B_pretrain", train=["omnicrack"], encoder=best_enc,
              imgsz=512, batch=8, epochs=8, lr=3e-4, aug=True)); log_result(rP)
# (B-2) 그 가중치로 Steelcrack 파인튜닝
rB = run(dict(name="E2B_finetune", train=["steelcrack"], encoder=best_enc,
              imgsz=512, batch=8, epochs=30, lr=1e-4, aug=True,
              init=rP["weights"], save_overlays=True)); log_result(rB)
print("도메인갭 회복:", round(rA["dice"],3), "→", round(rB["dice"],3))

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

[E2A_steelonly] train 3300장 | efficientnet-b0 imgsz512 aug=True


/tmp/ipykernel_3015/438889622.py:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(); lossf=DiceBCE()
/tmp/ipykernel_3015/438889622.py:49: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): loss=lossf(model(x),y)


  ep1/30 loss 0.4258
  ep2/30 loss 0.1991
  ep3/30 loss 0.1757
  ep4/30 loss 0.1704
  ep5/30 loss 0.1628
  ep6/30 loss 0.1576
  ep7/30 loss 0.1560
  ep8/30 loss 0.1530
  ep9/30 loss 0.1507
  ep10/30 loss 0.1451
  ep11/30 loss 0.1470
  ep12/30 loss 0.1446
  ep13/30 loss 0.1427
  ep14/30 loss 0.1413
  ep15/30 loss 0.1403
  ep16/30 loss 0.1451
  ep17/30 loss 0.1382
  ep18/30 loss 0.1375
  ep19/30 loss 0.1358
  ep20/30 loss 0.1346
  ep21/30 loss 0.1327
  ep22/30 loss 0.1347
  ep23/30 loss 0.1339
  ep24/30 loss 0.1343
  ep25/30 loss 0.1311
  ep26/30 loss 0.1320
  ep27/30 loss 0.1295
  ep28/30 loss 0.1269
  ep29/30 loss 0.1272
  ep30/30 loss 0.1251
  → Steelcrack Test: {'dice': 0.848, 'iou': 0.755, 'recall': 0.843, 'precision': 0.885, 'cldice': 0.901, 'n_pos': 530, 'n_neg': 0, 'fp_rate_on_clean': None}
logged → /content/drive/MyDrive/segmentation_experiment/experiment_results.csv
[E2B_pretrain] train 22158장 | efficientnet-b0 imgsz512 aug=True
  ep1/8 loss 0.3883
  ep2/8 loss 0.3463
  ep3/8 l

In [ ]:
# ====== CELL 7: E3 — 증강 유무 A/B ======================
rN = run(dict(name="E3_noaug", train=["steelcrack"], encoder=best_enc,
              imgsz=512, batch=8, epochs=25, lr=3e-4, aug=False)); log_result(rN)
rY = run(dict(name="E3_aug",   train=["steelcrack"], encoder=best_enc,
              imgsz=512, batch=8, epochs=25, lr=3e-4, aug=True));  log_result(rY)
print("증강 효과(Recall):", round(rN.get("recall",0),3), "→", round(rY.get("recall",0),3))

In [ ]:
# ====== CELL 8: E4 — 해상도 비교 (젯슨 속도 vs 성능) ======================
for sz in [320, 384, 512]:
    r = run(dict(name=f"E4_sz{sz}", train=["steelcrack"], encoder=best_enc,
                 imgsz=sz, batch=8, epochs=25, lr=3e-4, aug=True)); log_result(r)

In [ ]:
# ====== CELL 9: 결과 표로 보기 ======================
import pandas as pd
df = pd.read_csv(f"{DRIVE}/experiment_results.csv")
df = df.sort_values("recall", ascending=False)   # 안전상 Recall 우선 정렬
print(df[["name","dice","iou","recall","precision","cldice","fp_rate_on_clean"]].to_string(index=False))